# Zadanie 2: optymalizacja z ograniczeniami

Termin realizacji: 31 marca 2025

Wybierz funkcję testową wykorzystaną w zadaniu 1.

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Dodaj ograniczenie postaci $x_1^2 + x_2 + b = 0$ ze stałą $b$ dopasowaną w taki sposób, aby żadne minimum lokalne (przynajmniej w zakresie w którym losowany jest punkt początkowy) nie spełniają ograniczenia.
2. Zaimplementuj metodę funkcji kary do rozwiązania tego problemu.
3. Wylosuj 10 punktów z dziedziny przeszukiwania z tabelki. Dla każdego z nich przeprowadź 100 kroków optymalizacji metodą największego spadku ze stałym krokiem. Narysuj wykres zależności wartości funkcji optymalizowanej od kroku optymalizacji.
4. Przeprowadź procedurę dla kilku kroków. Spróbuj zilustrować brak zbieżności, szybką zbieżność i powolną zbieżność.

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Zamień metodę największego spadku na metodę gradientów sprzężonych.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Wykonaj benchmarking metody z użyciem `BenchmarkTools.jl`. Zanotuj czasy działania wywołań optymalizacji oraz liczbę alokacji. Spróbuj zoptymalizować działanie funkcji korzystając wymienionych tu rad: [Julia performance tips](https://docs.julialang.org/en/v1/manual/performance-tips/). W sprawozdaniu napisz jakie zmiany wykonane i jak wpłynęły na czas działania programu.


In [6]:
using LinearAlgebra
using Plots
using Random

# 1. Funkcja z zad.1

In [2]:

function f(vec)
    x, y = vec[1], vec[2]
    return 2 * x^2 - 1.05 * x^4 + x^6 / 6 + x*y + y^2
end

function f_grad(vec)
    x, y = vec[1], vec[2]
    return [4 * x - 4.2 * x^3 + x^5 + y, x + 2 * y]
end

f_grad (generic function with 1 method)

In [5]:
function steepest_gradient_descent(cost, grad, x0, α, γ; max_iter=1000, tol=1e-8)
    θ = copy(x0)
    f_values = []
    storage = zeros(length(θ))
    for i in 1:max_iter
        value_start = cost(θ)
        storage = grad(θ)
        norm_storage = norm(storage)
        if norm_storage == 0
            break
        end
        θ_new = θ - storage .* (α / norm_storage)
        value_stop = cost(θ_new)
        push!(f_values, value_start)
        if abs(value_stop - value_start) < tol
            break
        else
            θ = θ_new
            α *= γ
        end
    end
    return θ, f_values
end

steepest_gradient_descent (generic function with 1 method)

In [4]:
function rand_uniform(a, b)
    return rand() * (b-a) + a
end

# lista z 10 wylosowanymi punktami
points = [[rand_uniform(-5, 5), rand_uniform(-5, 5)] for i in 1:10]
points

10-element Vector{Vector{Float64}}:
 [0.9787965862764132, -0.20808121107781297]
 [0.8396259147666303, 3.2476996282213317]
 [-1.0301094520776601, 0.46554541381571557]
 [-4.798226130690301, 4.477638868963746]
 [0.4407390082053295, 4.131072460980064]
 [1.4365292193308044, -3.088287267631994]
 [2.2796400216279045, 0.17161404546402714]
 [2.4620963008037613, 0.28718029182853044]
 [-0.5818348179779633, -3.9204812908028366]
 [2.4822210095822497, 1.4232672550468104]

In [7]:
function penalty_method(f, p, x0, k_max, ρ=1.0, γ=2.0, atol=eps(eltype(x0)))
    x = x0
    for k in 1 : k_max
        x = steepest_gradient_descent(x -> f(x) + ρ*p(x), p, x0, 0.01, 0.99)
        ρ *= γ
        if p(x) < atol
            return x
        end
    end
    return x
end

    

penalty_method (generic function with 4 methods)

In [8]:
penalty_method(f, f_grad, [0, 0], 16)

MethodError: MethodError: no method matching eps(::Type{Int64})
The function `eps` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  eps(!Matched::Type{Dates.Time})
   @ Dates C:\Users\Adam\.julia\juliaup\julia-1.11.3+0.x64.w64.mingw32\share\julia\stdlib\v1.11\Dates\src\types.jl:454
  eps(!Matched::Type{Float16})
   @ Base float.jl:1048
  eps(!Matched::Type{Dates.DateTime})
   @ Dates C:\Users\Adam\.julia\juliaup\julia-1.11.3+0.x64.w64.mingw32\share\julia\stdlib\v1.11\Dates\src\types.jl:452
  ...
